In [1]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.tools import StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
class LoanEMIInput(BaseModel):
    principal: float = Field(
        gt=0,
        description="Loan principal amount in rupees, greater than zero"
    )

    annual_rate: float = Field(
        ge=0,
        le=30,
        description="Annual interest rate as a percentage, from 0 to 30"
    )

    tenure_months: int = Field(
        gt=0,
        description="Loan repayment tenure in months, greater than zero"
    )


print("Loan EMI input schema created.")
print(LoanEMIInput.model_json_schema())

Loan EMI input schema created.
{'properties': {'principal': {'description': 'Loan principal amount in rupees, greater than zero', 'exclusiveMinimum': 0, 'title': 'Principal', 'type': 'number'}, 'annual_rate': {'description': 'Annual interest rate as a percentage, from 0 to 30', 'maximum': 30, 'minimum': 0, 'title': 'Annual Rate', 'type': 'number'}, 'tenure_months': {'description': 'Loan repayment tenure in months, greater than zero', 'exclusiveMinimum': 0, 'title': 'Tenure Months', 'type': 'integer'}}, 'required': ['principal', 'annual_rate', 'tenure_months'], 'title': 'LoanEMIInput', 'type': 'object'}


In [3]:
def calculate_emi(
    principal: float,
    annual_rate: float,
    tenure_months: int
) -> str:
    """Calculate monthly loan EMI using principal, annual rate, and tenure."""

    # Validate all input values
    loan = LoanEMIInput(
        principal=principal,
        annual_rate=annual_rate,
        tenure_months=tenure_months
    )

    principal = loan.principal
    monthly_rate = loan.annual_rate / 12 / 100
    months = loan.tenure_months

    if monthly_rate == 0:
        emi = principal / months
    else:
        emi = (
            principal
            * monthly_rate
            * (1 + monthly_rate) ** months
            / ((1 + monthly_rate) ** months - 1)
        )

    total_payment = emi * months
    total_interest = total_payment - principal

    return (
        f"Loan principal: ₹{principal:,.2f}\n"
        f"Annual interest rate: {loan.annual_rate}%\n"
        f"Tenure: {months} months\n"
        f"Monthly EMI: ₹{emi:,.2f}\n"
        f"Total payment: ₹{total_payment:,.2f}\n"
        f"Total interest: ₹{total_interest:,.2f}"
    )


emi_tool = StructuredTool.from_function(
    func=calculate_emi,
    name="calculate_loan_emi",
    description=(
        "Calculate the monthly EMI, total repayment, and total interest "
        "for a loan. Requires principal in rupees, annual interest rate "
        "as a percentage, and tenure in months."
    ),
    args_schema=LoanEMIInput
)

print("StructuredTool created successfully.")
print("Tool name:", emi_tool.name)

StructuredTool created successfully.
Tool name: calculate_loan_emi


In [4]:
manual_result = emi_tool.invoke({
    "principal": 500000,
    "annual_rate": 9,
    "tenure_months": 24
})

print("Manual EMI calculation:")
print(manual_result)

Manual EMI calculation:
Loan principal: ₹500,000.00
Annual interest rate: 9.0%
Tenure: 24 months
Monthly EMI: ₹22,842.37
Total payment: ₹548,216.91
Total interest: ₹48,216.91


In [5]:
tools = [emi_tool]
llm_with_tools = llm.bind_tools(tools)

user_query = (
    "What's my monthly EMI for a 5 lakh rupee loan "
    "at 9% annual interest for 24 months? "
    "Please also tell me the total interest."
)

messages = [HumanMessage(content=user_query)]

response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nModel-suggested tool calls:")
print(response.tool_calls)

if not response.tool_calls:
    raise RuntimeError(
        "The model did not suggest a tool call. Run this cell again."
    )

User query:
What's my monthly EMI for a 5 lakh rupee loan at 9% annual interest for 24 months? Please also tell me the total interest.

Model-suggested tool calls:
[{'name': 'calculate_loan_emi', 'args': {'annual_rate': 9, 'principal': 500000, 'tenure_months': 24}, 'id': 'fc_627d3717-f1b5-4384-9b23-1c2ad5dbf8ff', 'type': 'tool_call'}]


In [6]:
tool_map = {
    "calculate_loan_emi": emi_tool
}

messages.append(response)
tool_results = []

for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    result = tool_map[tool_name].invoke(tool_args)
    tool_results.append(result)

    print("\nTool name:", tool_name)
    print("Tool arguments:", tool_args)
    print("\nExecuted tool result:")
    print(result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )

print("\nTool execution completed.")


Tool name: calculate_loan_emi
Tool arguments: {'annual_rate': 9, 'principal': 500000, 'tenure_months': 24}

Executed tool result:
Loan principal: ₹500,000.00
Annual interest rate: 9.0%
Tenure: 24 months
Monthly EMI: ₹22,842.37
Total payment: ₹548,216.91
Total interest: ₹48,216.91

Tool execution completed.


In [7]:
final_response = llm.invoke(messages)

print("Final answer from the model:")
print(final_response.content)

Final answer from the model:
**Loan Details**

| Item | Value |
|------|-------|
| Principal (P) | ₹ 5,00,000 |
| Annual interest rate (r) | 9 % |
| Tenure | 24 months (2 years) |
| Monthly interest rate (r / 12) | 0.75 % (0.0075) |

**EMI Calculation**

The standard EMI formula is  

\[
\text{EMI}= \frac{P \times r_{\text{monthly}} \times (1+r_{\text{monthly}})^{n}}{(1+r_{\text{monthly}})^{n}-1}
\]

where  

* \(P\) = loan amount (₹ 5,00,000)  
* \(r_{\text{monthly}}\) = 0.75 % = 0.0075  
* \(n\) = total number of monthly installments = 24  

Plugging the numbers in:

\[
\text{EMI}= \frac{5,00,000 \times 0.0075 \times (1+0.0075)^{24}}{(1+0.0075)^{24}-1}
          \approx \mathbf{₹ 22,842.37}
\]

**Total Repayment & Interest**

* **Total amount paid over 24 months** = EMI × 24  
  \[
  22,842.37 \times 24 \approx \mathbf{₹ 5,48,216.91}
  \]

* **Total interest paid** = Total repayment – Principal  
  \[
  5,48,216.91 - 5,00,000 = \mathbf{₹ 48,216.91}
  \]

---

### Summary
- **Monthly 

In [8]:
assert response.tool_calls, "No tool call was suggested."

assert any(
    call["name"] == "calculate_loan_emi"
    for call in response.tool_calls
), "The EMI calculator tool was not called."

assert tool_results, "No tool result was generated."

assert any(
    "Monthly EMI" in result
    for result in tool_results
), "EMI was not calculated."

assert final_response.content.strip(), "Final answer is empty."

print("Validation passed.")
print("The model suggested the EMI calculator tool.")
print("The tool was executed manually.")
print("The tool returned the EMI calculation.")
print("The model generated a final answer.")

Validation passed.
The model suggested the EMI calculator tool.
The tool was executed manually.
The tool returned the EMI calculation.
The model generated a final answer.


In [9]:
from pydantic import ValidationError

invalid_inputs = [
    {
        "principal": 0,
        "annual_rate": 9,
        "tenure_months": 24
    },
    {
        "principal": 500000,
        "annual_rate": 35,
        "tenure_months": 24
    },
    {
        "principal": 500000,
        "annual_rate": 9,
        "tenure_months": 0
    }
]

for index, invalid_input in enumerate(invalid_inputs, start=1):
    try:
        LoanEMIInput(**invalid_input)
        raise AssertionError(
            f"Invalid input {index} was incorrectly accepted."
        )
    except ValidationError:
        print(f"Invalid input {index} correctly rejected.")

print("All input validation tests passed.")

Invalid input 1 correctly rejected.
Invalid input 2 correctly rejected.
Invalid input 3 correctly rejected.
All input validation tests passed.


In [10]:
print("Lab 6 - Exercise 2 completed successfully.")
print("Tool method: StructuredTool with Pydantic args_schema.")
print("Principal must be greater than zero.")
print("Annual interest rate must be between 0 and 30.")
print("Tenure must be a positive number of months.")
print("Manual calculation, tool calling, and validation tested.")

Lab 6 - Exercise 2 completed successfully.
Tool method: StructuredTool with Pydantic args_schema.
Principal must be greater than zero.
Annual interest rate must be between 0 and 30.
Tenure must be a positive number of months.
Manual calculation, tool calling, and validation tested.
